In [1]:
#Task 1
# Data loader, weight tying ve başlangıç değerleri. Metni batch'lere
# bölen küçük bir data loader yaz ve metnin tamamında eğit. Weight tying'i
# ekle, kaç parametre kazandırdığını hesapla. Başlangıç değerlerini videodaki
# gibi ayarla: std 0.02, residual katmanlarda 1/√(2·n_layer) ile ölçekleme.
# Neden ölçeklediğimizi kendi cümlelerinle yaz. Geçen hafta ek görevde
# yaptıysan kısaca göstermen yeterli. (1:02:00 - 1:22:18)

In [2]:
#imports
import tiktoken
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import math

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        #batchwisely k q v projections for all heads
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        #output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1.0
        #regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        #mask
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size))


    def forward(self, x):
        B, T, C = x.size() #batch size, sequence length, embedding dims

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim = 2)
        k = k.view(B,T, self.n_head, C // self.n_head).transpose(1,2) #B, T, nh, hs to B, nh, T, hs
        q = q.view(B,T, self.n_head, C // self.n_head).transpose(1,2) #B, T, nh, hs to B, nh, T, hs
        v = v.view(B,T, self.n_head, C // self.n_head).transpose(1,2) #B, T, nh, hs to B, nh, T, hs
        #attention
        att = (q @ k.transpose(-2,-1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v #(B, nh, T, T) x (B, nh, T, hs) = (B, nh, T, hs)
        #taking all head outputs side by side
        y = y.transpose(1,2).contiguous().view(B,T,C) 
        #output projection
        y = self.c_proj(y)
        return y

In [4]:
class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate = "tanh") #nonlinearity
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1.0
    def forward(self,x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x
        

In [5]:
class Block(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self,x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x 

In [6]:
@dataclass
class GPTConfig:
    block_size: int= 1024
    vocab_size: int = 50257
    n_layer: int= 12 
    n_head: int= 12 
    n_embd: int= 768

class GPT(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
                    wte = nn.Embedding(config.vocab_size, config.n_embd), #token embds
                    wpe = nn.Embedding(config.block_size, config.n_embd), #positional embds
                    h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]), #hidden blocks
                    ln_f = nn.LayerNorm(config.n_embd),)) #final layernorm
        
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False) #final classifier  

        #weight tying // sharing
        self.transformer.wte.weight = self.lm_head.weight

        #param init
        self.apply(self._init_weights)

    #initialization parameter setting
    def _init_weights(self, module):
        #linear layer, embd -> N(0,0.02)
        #bias -> 0
        #layernorm parameters already comes from pytorch
        std = 0.02

        if hasattr(module, "NANOGPT_SCALE_INIT"):
            std *= (2 * self.config.n_layer) ** -0.5
        if isinstance(module, nn.Linear): 
            torch.nn.init.normal_(module.weight, mean = 0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean = 0.0, std =std)
        
    def forward(self, idx, targets=None):
        # idx is of shape (B, T)
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"
        #first embeddings token and positional
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device) # shape (T)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (T, n_embd)
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (B, T, n_embd)
        x = tok_emb + pos_emb
        #going for blocks
        for block in self.transformer.h:
            x = block(x)
        #final layernorm and the classifier
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)
        loss = None

        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),targets.view(-1))
        return logits, loss
    
    @classmethod
    def from_pretrained(cls, model_type):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

In [7]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = GPT(GPTConfig())
model.to(device)

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [8]:
#how to load data to model
class DataLoaderLite:
    def __init__(self, B, T):
        self.B = B
        self.T = T
    #tokenizing them
        with open("input.txt", "r") as f:
            text = f.read()
    
        enc = tiktoken.get_encoding("gpt2")  #gpt2's encoder
        tokens = enc.encode(text)            #encoding data
        self.tokens = torch.tensor(tokens)
    
        print(f"loaded {len(self.tokens)} tokens")
        print(f"1 epoch = {len(self.tokens) // (B*T)} batches")
    
    #state
        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        #as usual. gather B*T + 1 data. first B * T to input. shift input with one data point and make them target   
        buf = self.tokens[self.current_position: self.current_position + (B * T) + 1] #taking batch + 1 
        x = (buf[:-1]).view(B,T) #till last one
        y = (buf[1:]).view(B,T)  #second 2 last
        #advancing position in big token tensor
        self.current_position += B*T
        #prevention before hitting last data piece
        if self.current_position + (B * T + 1) > len(self.tokens):
            self.current_position = 0 
        
        return x,y

In [9]:
train_loader = DataLoaderLite(4,32)

loaded 338025 tokens
1 epoch = 2640 batches


In [10]:
#optimizing loop
optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-4)

for i in range(50):
    x,y = train_loader.next_batch()
    x,y = x.to(device), y.to(device)
    optimizer.zero_grad()
    logits, loss = model(x,y)
    loss.backward()
    optimizer.step()
    print(f"step {i}, loss: {loss.item()}")


step 0, loss: 10.98766803741455
step 1, loss: 9.759376525878906
step 2, loss: 9.040225982666016
step 3, loss: 9.195720672607422
step 4, loss: 8.780526161193848
step 5, loss: 8.423677444458008
step 6, loss: 8.922867774963379
step 7, loss: 8.684030532836914
step 8, loss: 8.116095542907715
step 9, loss: 7.930649757385254
step 10, loss: 8.264776229858398
step 11, loss: 7.454305648803711
step 12, loss: 7.8627028465271
step 13, loss: 7.464719772338867
step 14, loss: 7.541532516479492
step 15, loss: 7.349627494812012
step 16, loss: 7.401242256164551
step 17, loss: 8.255971908569336
step 18, loss: 7.181852340698242
step 19, loss: 7.81443977355957
step 20, loss: 7.496609687805176
step 21, loss: 7.7972917556762695
step 22, loss: 6.439445972442627
step 23, loss: 6.883656024932861
step 24, loss: 6.863645076751709
step 25, loss: 6.695368766784668
step 26, loss: 6.8839640617370605
step 27, loss: 7.652801990509033
step 28, loss: 7.192878246307373
step 29, loss: 6.91972541809082
step 30, loss: 6.95838

In [13]:
model.transformer.wte.weight.numel() #saved paramaters count

38597376